# Data Audit Notebook

Sanity checks for SQLite ingestion outputs with a focus on small-table previews and key field quality checks.

In [18]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path('../data/interim/books.db')
assert DB_PATH.exists(), f'Missing database at {DB_PATH.resolve()}'
conn = sqlite3.connect(DB_PATH)
print('Connected to:', DB_PATH.resolve())

Connected to: /Users/zacurbiztondo/literary-analysis/data/interim/books.db


In [2]:
# 1) List tables
tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """,
    conn
)
tables

,table_name
0,nyt_entries
1,openlibrary_enrichment


In [20]:
pd.read_sql_query(
    """
    SELECT COUNT(*)
    FROM openlibrary_enrichment
    """,
    conn
)

,COUNT(*)
0,856


In [21]:
pd.read_sql_query(
    """
    SELECT COUNT(*)
    FROM openlibrary_enrichment
    WHERE work_key IS NOT NULL
    """,
    conn
)

,COUNT(*)
0,568


In [ ]:
pd.read_sql_query(
    """
    SELECT COUNT(*)
    FROM nyt_entries
    """,
    conn
)

,COUNT(*)
0,3150


In [12]:
pd.read_sql_query(
    """
    SELECT COUNT(*)
    FROM openlibrary_enrichment
    WHERE Subjects IS NOT NULL
    """,
    conn
)

,COUNT(*)
0,856


In [ ]:
nyt_ol = pd.read_csv('../data/processed/nyt_openlibrary_joined.csv')

np.int64(2069)

In [8]:

nyt_ol['subjects'].isna().sum()

np.int64(2069)

In [ ]:
len(nyt_ol)

3150

In [16]:
nyt_ol[nyt_ol['title'] == "THE GIRL WHO FELL BENEATH THE SEA"]

,id,list_name,published_date,rank,weeks_on_list,title,author,publisher,isbn13,isbn10,description,created_at,work_key,subjects,subject_places,ol_description,last_error
1675,1676,Young Adult Paperback,2025-07-06,6,0,THE GIRL WHO FELL BENEATH THE SEA,Axie Oh,Square Fish,9781250866097,NaN,NaN,2026-03-02 00:49:02,/works/OL25219259W,nyt:young-adult-hardcover=2022-03-13|New York ...,NaN,Deadly storms have ravaged Mina’s homeland for...,NaN


In [ ]:
# 2) Row counts per table
counts = []
for t in tables['table_name']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    counts.append({'table_name': t, 'row_count': n})
counts_df = pd.DataFrame(counts).sort_values('table_name').reset_index(drop=True)
counts_df

In [23]:
# 3) ISBN13 and work_key format checks in openlibrary_enrichment
quality_sql = """
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN isbn13 GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]' THEN 1 ELSE 0 END) AS isbn13_valid_rows,
  SUM(CASE WHEN isbn13 NOT GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]' THEN 1 ELSE 0 END) AS isbn13_invalid_rows,
  SUM(CASE WHEN work_key IS NULL OR TRIM(work_key) = '' THEN 1 ELSE 0 END) AS work_key_blank_rows,
  SUM(CASE WHEN work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key GLOB '/works/*' THEN 1 ELSE 0 END) AS work_key_valid_rows,
  SUM(CASE WHEN work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key NOT GLOB '/works/*' THEN 1 ELSE 0 END) AS work_key_invalid_rows
FROM openlibrary_enrichment
"""
pd.read_sql_query(quality_sql, conn)

,total_rows,isbn13_valid_rows,isbn13_invalid_rows,work_key_blank_rows,work_key_valid_rows,work_key_invalid_rows
0,856,848,8,288,568,0


In [28]:
# 4) Show malformed isbn13/work_key rows for inspection
bad_rows_sql = """
SELECT isbn13, work_key, last_error, last_checked_at
FROM openlibrary_enrichment
WHERE isbn13 NOT GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]'
   OR (work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key NOT GLOB '/works/*')
ORDER BY isbn13
"""
pd.read_sql_query(bad_rows_sql, conn)

,isbn13,work_key,last_error,last_checked_at
0,A00B01K4VZYGY,None,edition_not_found,2026-03-02T01:03:45+00:00
1,DBKACX0234484,None,edition_not_found,2026-03-02T01:03:45+00:00
2,DBKACX0251720,None,edition_not_found,2026-03-02T01:03:45+00:00
3,DBKACX0257961,None,edition_not_found,2026-03-02T01:03:45+00:00
4,DBKACX0286289,None,edition_not_found,2026-03-02T01:03:45+00:00
5,DBKADBL022168,None,edition_not_found,2026-03-02T01:03:45+00:00
6,DBKADBL059878,None,edition_not_found,2026-03-02T01:03:45+00:00
7,DBKADBL063268,None,edition_not_found,2026-03-02T01:03:45+00:00


In [29]:
# 5) Preview small tables in full, otherwise head(20)
def preview_table(table_name: str, small_threshold: int = 100):
    n = conn.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
    if n <= small_threshold:
        q = f'SELECT * FROM {table_name}'
    else:
        q = f'SELECT * FROM {table_name} LIMIT 20'
    df = pd.read_sql_query(q, conn)
    print(f'\n=== {table_name} (rows={n}) ===')
    return df

for t in tables['table_name']:
    display(preview_table(t))


=== nyt_entries (rows=3150) ===


,id,list_name,published_date,rank,weeks_on_list,title,author,publisher,isbn13,isbn10,description,created_at
0,1,Combined Print & E-Book Fiction,2024-12-29,1,8,JAMES,Percival Everett,Doubleday,9780385550369,,A reimagining of “Adventures of Huckleberry Fi...,2026-03-02 00:46:04
1,2,Combined Print & E-Book Fiction,2024-12-29,2,2,WIND AND TRUTH,Brandon Sanderson,Tor,9781250319180,,The fifth book in the Stormlight Archive serie...,2026-03-02 00:46:04
2,3,Combined Print & E-Book Fiction,2024-12-29,3,5,WICKED,Gregory Maguire,Morrow,9780062852847,,A misunderstood girl named Elphaba is declared...,2026-03-02 00:46:04
3,4,Combined Print & E-Book Fiction,2024-12-29,4,45,THE WOMEN,Kristin Hannah,St. Martin's,9781250178633,,A nurse follows her brother to serve during th...,2026-03-02 00:46:04
4,5,Combined Print & E-Book Fiction,2024-12-29,5,73,FOURTH WING,Rebecca Yarros,Red Tower,9781649377371,,Violet Sorrengail is urged by the commanding g...,2026-03-02 00:46:04
5,6,Combined Print & E-Book Fiction,2024-12-29,6,3,THE HOUSE OF CROSS,James Patterson,"Little, Brown",9780316402682,,The 33rd book in the Alex Cross series. Three ...,2026-03-02 00:46:04
6,7,Combined Print & E-Book Fiction,2024-12-29,7,7,THE GOD OF THE WOODS,Liz Moore,Riverhead,9780593418918,,When a 13-year-old girl disappears from an Adi...,2026-03-02 00:46:04
7,8,Combined Print & E-Book Fiction,2024-12-29,8,5,THE WEDDING PEOPLE,Alison Espach,Holt,9781250899569,,A woman who is down on her luck forms an unexp...,2026-03-02 00:46:04
8,9,Combined Print & E-Book Fiction,2024-12-29,9,4,THE FROZEN RIVER,Ariel Lawhon,Vintage,9780593312070,,"In Maine, 1789, a midwife seeks to uncover the...",2026-03-02 00:46:04
9,10,Combined Print & E-Book Fiction,2024-12-29,10,36,A COURT OF THORNS AND ROSES,Sarah J. Maas,Bloomsbury,9781635575569,,"After killing a wolf in the woods, Feyre is ta...",2026-03-02 00:46:04



=== openlibrary_enrichment (rows=856) ===


,isbn13,work_key,subjects,subject_places,description,last_error,last_checked_at
0,9780008725716,NaN,,,NaN,edition_not_found,2026-03-02T00:54:47+00:00
1,9780008728090,/works/OL42395220W,Romance|Christmas|Fiction|Holiday|Contemporary...,,**From the international bestselling author of...,NaN,2026-03-02T00:54:47+00:00
2,9780061968358,NaN,,,NaN,edition_not_found,2026-03-02T00:54:47+00:00
3,9780062200600,/works/OL42552307W,Horror|fiction|time jump,,"Arthur Oakes is a reader, a dreamer, and a stu...",NaN,2026-03-02T00:54:47+00:00
4,9780062300553,/works/OL17357665W,Mountain people|Family|Social mobility|Working...,United States|Appalachian Region|Kentucky,From a former marine and Yale Law School gradu...,NaN,2026-03-02T00:54:47+00:00
5,9780062406682,/works/OL44220771W,,,"When he is eight years old, Alfie Logan discov...",NaN,2026-03-02T00:54:47+00:00
6,9780062407801,/works/OL18819818W,Negotiation|Negotiation in business|Business c...,,"""A negotiation guide from a former FBI Hostage...",NaN,2026-03-02T00:54:47+00:00
7,9780062477521,NaN,,,NaN,edition_not_found,2026-03-02T00:54:47+00:00
8,9780062852847,/works/OL34638280W,,,NaN,NaN,2026-03-02T00:54:47+00:00
9,9780062863102,/works/OL43273995W,,,The extraordinary life of Cher can be told by ...,NaN,2026-03-02T00:54:47+00:00


In [17]:
# 6) Optional: close connection when done
conn.close()